# 03 — Model Training: Walk-Forward XGBoost & Exponential Ensemble

**Series:** Piccolo ML Options Strategy Research  
**Depends on:** `02_feature_engineering.ipynb`


## 1. Overview

This notebook covers:

1. Walk-forward validation methodology and its advantages over k-fold CV
2. Running `train_walkforward()` from `ml_signal_engine`
3. Per-fold metrics: accuracy, macro-F1, confusion matrix
4. Ensemble construction: exponential recency weighting (ALPHA=0.7)
5. Ensemble vs single-fold comparison
6. Confidence threshold tuning
7. Feature importance stability across folds


## 2. Environment Setup


In [ ]:
%matplotlib inline
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
import warnings
warnings.filterwarnings("ignore")

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import src.piccolo.config_strategy as cfg
from src.piccolo.ml_signal_engine import (
    load_feature_table_spy,
    build_path_labels,
    build_ml_table,
    train_walkforward,
    add_signal_columns,
    INV_LABEL_MAP,
)

print("Imports OK")
print(f"  N_TRAIN_MONTHS = {cfg.N_TRAIN_MONTHS}")
print(f"  N_TEST_MONTHS  = {cfg.N_TEST_MONTHS}")


## 3. Walk-Forward Methodology

Walk-forward validation (also called *rolling-window* or *time-series CV*)
is the correct approach for any time-dependent strategy.  Standard k-fold CV
leaks future data into training, which inflates apparent performance.

```
Time ──────────────────────────────────────────────────────►

Fold 1:  [====TRAIN (N_TRAIN_MONTHS)====][TEST (N_TEST_MONTHS)]
Fold 2:        [====TRAIN====][TEST]
Fold 3:              [====TRAIN====][TEST]
...

Each fold shifts forward by N_TEST_MONTHS.
The training window ALWAYS precedes the test window in time.
```

**Key design choices:**

| Choice | Rationale |
|--------|----------|
| Rolling (not expanding) window | Markets change; a 5-year-old regime should not equally weight with last month |
| `N_TRAIN_MONTHS` | Enough history for XGBoost to learn structure, short enough to stay relevant |
| `N_TEST_MONTHS` | Test period long enough for statistical meaning, short enough to be realistic |
| Exponential ensemble (α=0.7) | More recent folds get higher weight — recency matters in non-stationary markets |


In [ ]:
# ── Load data & prepare ML table ────────────────────────────────────────────
feat_df = load_feature_table_spy()
feat_df = build_path_labels(feat_df)
ml_df, feature_cols = build_ml_table(feat_df)

print(f"ML table shape:  {ml_df.shape}")
print(f"Features used:   {len(feature_cols)}")
print(f"Date range:      {ml_df['quote_date'].min().date()} → {ml_df['quote_date'].max().date()}")
print(f"Label dist:\n{ml_df['label'].value_counts().sort_index().to_string()}")


## 4. Run Walk-Forward Training


In [ ]:
# ── Train the walk-forward ensemble ──────────────────────────────────────────
# train_walkforward returns:
#   results_df   : DataFrame with per-row predictions, probabilities, fold_id
#   fold_models  : {fold_id: XGBClassifier}
#   fold_scalers : {fold_id: StandardScaler}

print("Starting walk-forward training...")
results_df, fold_models, fold_scalers = train_walkforward(ml_df, feature_cols)

print(f"\nResults shape:    {results_df.shape}")
print(f"Folds trained:    {len(fold_models)}")
print(f"Test date range:  {results_df['quote_date'].min().date()} → {results_df['quote_date'].max().date()}")
print(f"\nColumns available:\n  {list(results_df.columns)}")


## 5. Per-Fold Metrics


In [ ]:
# ── Per-fold accuracy and macro-F1 ───────────────────────────────────────────
# results_df has a 'fold_id' column — group by it to get per-fold metrics.

fold_metrics = []
for fid in sorted(results_df["fold_id"].unique()):
    sub = results_df[results_df["fold_id"] == fid]
    y_true = sub["label"].values
    y_pred = sub["pred_dir"].values  # single-fold prediction (mapped back to -1,0,1)
    y_pred_ens = sub["pred_dir_ens"].values  # ensemble prediction

    acc      = accuracy_score(y_true, y_pred)
    f1       = f1_score(y_true, y_pred, average="macro", zero_division=0)
    acc_ens  = accuracy_score(y_true, y_pred_ens)
    f1_ens   = f1_score(y_true, y_pred_ens, average="macro", zero_division=0)

    fold_metrics.append({
        "fold": fid,
        "rows": len(sub),
        "train_start": sub["train_start"].iloc[0].date(),
        "test_start":  sub["test_start"].iloc[0].date(),
        "test_end":    sub["test_end"].iloc[0].date(),
        "accuracy":    acc,
        "macro_f1":    f1,
        "acc_ens":     acc_ens,
        "f1_ens":      f1_ens,
    })

fold_df = pd.DataFrame(fold_metrics)
print(fold_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
x = fold_df["fold"]
ax.bar(x - 0.2, fold_df["accuracy"], width=0.35, label="Single-Fold Acc", color="#20808D", alpha=0.85)
ax.bar(x + 0.2, fold_df["acc_ens"],  width=0.35, label="Ensemble Acc", color="#A84B2F", alpha=0.85)
ax.axhline(fold_df["accuracy"].mean(), color="#20808D", linestyle="--", alpha=0.5,
           label=f"Avg Single ({fold_df['accuracy'].mean():.3f})")
ax.axhline(fold_df["acc_ens"].mean(),  color="#A84B2F", linestyle="--", alpha=0.5,
           label=f"Avg Ensemble ({fold_df['acc_ens'].mean():.3f})")
ax.set_xlabel("Fold ID")
ax.set_ylabel("Accuracy")
ax.set_title("Per-Fold Accuracy: Single-Fold vs Ensemble", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("per_fold_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Confusion matrices (first fold, last fold, all folds combined) ────────────
def plot_cm(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred, labels=[-1, 0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Down", "Flat", "Up"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title, fontsize=10, fontweight="bold")

fold_ids = sorted(results_df["fold_id"].unique())

if len(fold_ids) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # First fold (ensemble predictions)
    first = results_df[results_df["fold_id"] == fold_ids[0]]
    plot_cm(first["label"], first["pred_dir_ens"], f"Fold {fold_ids[0]} (earliest)", axes[0])

    # Last fold
    last = results_df[results_df["fold_id"] == fold_ids[-1]]
    plot_cm(last["label"], last["pred_dir_ens"], f"Fold {fold_ids[-1]} (most recent)", axes[1])

    # Combined
    plot_cm(results_df["label"], results_df["pred_dir_ens"], "All Folds (Ensemble)", axes[2])

    plt.suptitle("Confusion Matrices (Ensemble Predictions)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nCombined classification report (ensemble):")
    print(classification_report(
        results_df["label"], results_df["pred_dir_ens"],
        target_names=["Down", "Flat", "Up"], zero_division=0
    ))


## 6. Ensemble Weighting

The ensemble combines per-fold predictions using **exponential recency
weighting**.  The most recent fold gets the highest weight; each earlier fold
is discounted by a factor of `ALPHA`:

```
weight[k] = ALPHA^(N_folds - 1 - k)   (k = 0 … N_folds-1, last fold is most recent)
weights are then normalised to sum to 1.
```

A higher `ALPHA` (closer to 1.0) gives more equal weight across folds.
A lower `ALPHA` (closer to 0) concentrates weight on the most recent fold.
`ALPHA = 0.7` is the default — it provides a good balance between stability
and recency responsiveness.


In [ ]:
# ── Visualise ensemble weights ────────────────────────────────────────────────
# Reproduce the weight calculation from ml_signal_engine.train_walkforward
ALPHA = cfg.ALPHA  # must match ml_signal_engine
all_fold_ids = sorted(fold_models.keys())
K = len(all_fold_ids)

raw_weights = np.array([ALPHA ** (K - 1 - idx) for idx in range(K)])
fold_weights = raw_weights / raw_weights.sum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(K), fold_weights, tick_label=[f"Fold {fid}" for fid in all_fold_ids],
       color="#20808D", edgecolor="white")
ax.set_xlabel("Fold")
ax.set_ylabel("Ensemble Weight")
ax.set_title(f"Exponential Ensemble Weights (ALPHA={ALPHA}, {K} folds)", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("ensemble_weights.png", dpi=150, bbox_inches="tight")
plt.show()

print("Weights (normalised):")
for fid, w in zip(all_fold_ids, fold_weights):
    print(f"  Fold {fid:2d}: {w:.4f}")


## 7. Confidence Threshold Tuning

The strategy only fires signals when the ensemble predicted-class probability
exceeds `CONF_THRESHOLD_UP` (for Up signals) or `CONF_THRESHOLD_DOWN` (for
Down signals).  Higher thresholds → fewer but higher-quality signals.

The trade-off is **coverage vs precision**.  This section sweeps thresholds
and plots the precision/coverage curve.


In [ ]:
# ── Threshold sweep ───────────────────────────────────────────────────────────
# results_df has columns: proba_up_ens, proba_down_ens, proba_flat_ens, label
prob_down = results_df["proba_down_ens"].values
prob_up   = results_df["proba_up_ens"].values
y_true    = results_df["label"].values

thresholds = np.arange(0.30, 0.80, 0.05)
rows = []
for thr in thresholds:
    mask_up   = prob_up >= thr
    mask_down = prob_down >= thr

    prec_up   = (y_true[mask_up] == 1).mean()    if mask_up.sum() > 0 else np.nan
    prec_down = (y_true[mask_down] == -1).mean()  if mask_down.sum() > 0 else np.nan
    cov_up    = mask_up.mean()
    cov_down  = mask_down.mean()

    rows.append({
        "threshold": thr,
        "up_precision":   prec_up,
        "down_precision": prec_down,
        "up_coverage":    cov_up,
        "down_coverage":  cov_down,
    })

thr_df = pd.DataFrame(rows)
print(thr_df.to_string(index=False, float_format="{:.3f}".format))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, direction, prec_col, cov_col, color, thr_val in [
    (axes[0], "Up",   "up_precision",   "up_coverage",   "#437A22", cfg.CONF_THRESHOLD_UP),
    (axes[1], "Down", "down_precision", "down_coverage", "#A13544", cfg.CONF_THRESHOLD_DOWN),
]:
    ax2 = ax.twinx()
    ax.plot(thr_df["threshold"], thr_df[prec_col], "o-", color=color, label="Precision")
    ax2.plot(thr_df["threshold"], thr_df[cov_col], "s--", color="#7A7974", label="Coverage")
    ax.axvline(thr_val, color="black", linestyle=":", label=f"Current ({thr_val})")
    ax.set_xlabel("Confidence Threshold")
    ax.set_ylabel(f"{direction} Precision", color=color)
    ax2.set_ylabel("Coverage", color="#7A7974")
    ax.set_title(f"{direction} Signal: Precision vs Coverage", fontsize=11, fontweight="bold")
    ax.legend(loc="upper left")
    ax2.legend(loc="upper right")
    ax.grid(alpha=0.3)

plt.suptitle("Confidence Threshold Sweep", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("threshold_sweep.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Feature Importance Across Folds


In [ ]:
# ── Extract feature importances from each fold's XGBClassifier ────────────────
fi_list = []
for fid in sorted(fold_models.keys()):
    model = fold_models[fid]
    if hasattr(model, "feature_importances_"):
        fi = model.feature_importances_
    else:
        # If loaded from booster JSON, get importance from booster
        booster = model.get_booster()
        score = booster.get_score(importance_type="weight")
        fi = np.array([score.get(f"f{i}", 0) for i in range(len(feature_cols))])
        if fi.sum() > 0:
            fi = fi / fi.sum()
    fi_list.append(fi)

if len(fi_list) > 0:
    fi_arr = np.array(fi_list)
    fi_mean = fi_arr.mean(axis=0)
    fi_std  = fi_arr.std(axis=0)

    fi_df = pd.DataFrame({
        "feature": feature_cols[:len(fi_mean)],
        "mean_importance": fi_mean,
        "std_importance": fi_std
    }).sort_values("mean_importance", ascending=False).reset_index(drop=True)

    print("Mean feature importances across folds:")
    print(fi_df.to_string(index=False, float_format="{:.4f}".format))

    fig, ax = plt.subplots(figsize=(10, 7))
    y_pos = range(len(fi_df))
    ax.barh(list(y_pos), fi_df["mean_importance"], xerr=fi_df["std_importance"],
            color="#20808D", edgecolor="white", alpha=0.85, capsize=3)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(fi_df["feature"])
    ax.invert_yaxis()
    ax.set_xlabel("Mean XGBoost Feature Importance (± std across folds)")
    ax.set_title("Feature Importance Stability Across Walk-Forward Folds",
                 fontsize=13, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig("feature_importance_folds.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Could not extract feature importances from fold models.")


## 9. Findings & Notes

> **TODO:** Fill in after running this notebook.

| Item | Result | Notes |
|------|--------|-------|
| Number of folds | _paste here_ | |
| Mean fold accuracy (single) | _paste here_ | |
| Mean fold accuracy (ensemble) | _paste here_ | |
| Mean fold macro-F1 | _paste here_ | |
| Best Up threshold (precision) | _paste here_ | |
| Best Down threshold (precision) | _paste here_ | |
| Top 3 features by importance | _paste here_ | |

## 10. Next

Proceed to [04_backtest_performance.ipynb](04_backtest_performance.ipynb).
